<a href="https://colab.research.google.com/github/JawBlade/Heart-Disease-Neural-Network/blob/main/Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Learning How Neural Networks Work

I wanted to learn and create my own neural network from **scratch**. I started my journey with [this article](https://blog.stackademic.com/learn-to-build-a-neural-network-from-scratch-yes-really-cac4ca457efc) by Aadil Mallick.

---

# Vectors & Matrix Math
The article introduced me to **Vectorization**, which was a totally new concept for me. To really understand the "why" behind it, I watched [3Blue1Brown’s Matrix Multiplication](https://www.youtube.com/watch?v=XkY2DOUCWMU).

It gave me a deeper understanding of how multiplying matrices isn't just a list of steps, but a **composition of linear transformations**. Seeing how space "squishes" and "morphs" visually helped me understand how data flows through layers.

# Challenge
The hardest part was jumping back into math after not having math since sophmore year of high school (I'm currently a Senior) . Moving from basic intuition into:
* **The Chain Rule:** $f'(g(x))$
* **Partial Derivatives:** $\frac{\partial C}{\partial w}$

It was a steep learning curve, but essential for understanding how the network actually learns.

# Finally Clicked
Everything clicked when I watched the [3Blue1Brown Backpropagation video](https://www.youtube.com/watch?v=Ilg3gGewQ5U). I finally understood how **Gradient Descent** works: we minimize the cost by finding the partial derivative of every weight and bias to determine exactly how to turn the "math knobs." Which are every weight and bias. That we can either say increase or decrease. Each valuse will affect the cost. How do we find how much a value changes the cost? We find the derivative of the cost with respect to that value($\frac{\partial C}{\partial w^{[L]}})$ or ($\frac{\partial C}{\partial b^{[L]}}$). We then know how it will affect our ouput and adjust it as needed.

#Project Goal
---
This notebook documents my journey of building a Neural Network from scratch with an **11-12-8-1** architecture.

## The Mission
I am building a model to detect heart disease risk using a [Kaggle dataset](https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset/data) containing records for 70,000 patients (Age, BP, Cholesterol, etc.). While I'm aware this model isn't for clinical use, the goal is to prove I can master the math and the implementation without relying on high-level libraries or AI-generated code.

## The Technical Requirements
* **Standardization:** Scaling features so the math stays stable.
* **Feed-Forward Process (FFP):** Using **ReLU** for hidden layers and **Sigmoid** for the final output.
* **Vectorized Backpropagation:** Calculating gradients across all 70k samples (or mini-batches) efficiently.
* **Suffle Data and do Batches of 512**: I think 512 people per iteration to make the computation easier for the computer.
* **Pure Logic:** I am writing the code myself to ensure I can explain every line, from matrix dimensions to the chain rule.

I did use AI for these descriptions but I did write it all down in my own words before letting it shorten and make it more concise.

#Step 1: Setting Up the Architecture
---
In the article he initialised the model with a array so I will do the same thing to make a 11-12-8-1 NN.

In [127]:
n = [11, 12, 8, 1]
print("Input layer size", n[0], 'aka', n[0], 'features')
print("Hidden Layer 1 size", n[1])
print("Hidden Layer 2 size", n[2])
print("Output Layer", n[3])

Input layer size 11 aka 11 features
Hidden Layer 1 size 12
Hidden Layer 2 size 8
Output Layer 1


Now we have to generate the random weights and biases.

In [128]:
import numpy as np

# Xavier/Glorot Initialization for weights
W1 = np.random.randn(n[1], n[0]) * np.sqrt(2. / (n[0] + n[1]))
W2 = np.random.randn(n[2], n[1]) * np.sqrt(2. / (n[1] + n[2]))
W3 = np.random.randn(n[3], n[2]) * np.sqrt(2. / (n[2] + n[3]))

b1 = np.random.randn(n[1], 1)
b2 = np.random.randn(n[2], 1)
b3 = np.random.randn(n[3], 1)

# Just Double Checking their the right Dimensions
print("Weights for layer 1 shape:", W1.shape)
print("Weights for layer 2 shape:", W2.shape)
print("Weights for layer 3 shape:", W3.shape)
print("bias for layer 1 shape:", b1.shape)
print("bias for layer 2 shape:", b2.shape)
print("bias for layer 3 shape:", b3.shape)

Weights for layer 1 shape: (12, 11)
Weights for layer 2 shape: (8, 12)
Weights for layer 3 shape: (1, 8)
bias for layer 1 shape: (12, 1)
bias for layer 2 shape: (8, 1)
bias for layer 3 shape: (1, 1)


#Step 2: Add The Input Data and Labels
---
Here is already where I deviate from the turorial. He hard codeed the inputs and he only has 2 input nodes. I'm going to use **csv reader** (a built-in python module) which, I learned from taking CS50, it will help me parse the data from my csv file into a numpy array.

In [129]:
import csv
import numpy as np

data = []
labels = [] # Can also be called y

with open('/content/Heart_Disease_Dataset_70k.csv', mode='r') as file:
    reader = csv.DictReader(file, dialect='excel')
    for row in reader:
      age = float(row['age'])
      gender =      float(row['gender'])
      height =      float(row['height'])
      weight =      float(row['weight'])
      ap_hi  =      float(row['ap_hi'])
      ap_lo  =      float(row['ap_lo'])
      cholesterol = float(row['cholesterol'])
      gluc   =  float(row['glucose'])
      smoke  =  float(row['smokeing'])
      alco   =  float(row['alcohol intake'])
      active =  float(row['active'])

      cardio =  float(row['cardio']) # Labels

      features = [age, gender, height, weight, ap_hi, ap_lo, cholesterol, gluc, smoke, alco, active]

      data.append(features)
      labels.append(cardio)

m = len(data)

# input = x, which I will use for equations, could also be called features
input = np.array(data).T
labels = np.array(labels).reshape(n[3], m)

# Notation = n^[0] x m
print(input.shape)

# Notation = n^[3] x m
print(labels.shape)

(11, 70000)
(1, 70000)


# Step 3: Setup Sigmoid Function And Begin FFP (Feed Forward Process)

In [130]:
# this was mentioned at the very end but I did come across this before on my own when learning this.
# I think I will define it here since the sigmoid function is also here.
# thanks to https://codesignal.com/learn/courses/introduction-to-data-cleaning-with-python/lessons/standardizing-and-normalizing-data-in-python
# for a solution using a libray, I was struggling to do it from scratch

import pandas as pd
from sklearn.preprocessing import StandardScaler

def standardize(arr):
    scaler = StandardScaler()
    standardized_data = scaler.fit_transform(arr)
    return standardized_data


def sigmoid(arr):
  return 1 / (1 + np.exp(-1 * arr))

In [131]:
def feed_forward(A0):
  m = A0.shape[1] # Use A0.shape[1] for m as A0 is (n_features, m)

  # Standardize the input data
  # Transpose A0 for standardization (n_features, m) -> (m, n_features)
  A0_standardized_T = standardize(A0.T)
  # Transpose back to (n_features, m)
  A0_standardized = A0_standardized_T.T

  # layer 1 calculations
  Z1 = W1 @ A0_standardized + b1
  A1 = sigmoid(Z1)

  # layer 2 calculations
  Z2 = W2 @ A1 + b2
  A2 = sigmoid(Z2)

  # layer 3 calculations
  Z3 = W3 @ A2 + b3
  A3 = sigmoid(Z3)

  cache = {
      "A0": A0_standardized, # Store the standardized A0 in cache
      "A1": A1,
      "A2": A2
  }

  return A3, cache #A3 is y_hat

In [132]:
print(input.shape) # prints out (1, 10)
y_hat = feed_forward(input)
print(y_hat) #output of model

(11, 70000)
(array([[0.81700649, 0.819903  , 0.81668178, ..., 0.82835723, 0.81450332,
        0.81857795]]), {'A0': array([[-3.51440749, -3.48968345, -3.48198252, ...,  1.71168771,
         1.71533552,  1.72019926],
       [ 1.36405487, -0.73310834, -0.73310834, ...,  1.36405487,
        -0.73310834,  1.36405487],
       [ 1.29606378, -0.65276302,  1.29606378, ...,  0.32165038,
         1.66146881,  1.29606378],
       ...,
       [-0.31087913, -0.31087913, -0.31087913, ..., -0.31087913,
        -0.31087913, -0.31087913],
       [-0.23838436, -0.23838436, -0.23838436, ..., -0.23838436,
        -0.23838436, -0.23838436],
       [ 0.49416711,  0.49416711,  0.49416711, ..., -2.02360695,
         0.49416711,  0.49416711]]), 'A1': array([[0.12698584, 0.38378984, 0.17905248, ..., 0.35111741, 0.17366725,
        0.22297549],
       [0.69311152, 0.57367466, 0.59507643, ..., 0.64632859, 0.42287902,
        0.65003367],
       [0.37705148, 0.53835637, 0.52274768, ..., 0.01170341, 0.05613363,
   

# Step 4: Calculating cost

In [133]:
def cost(y_hat, y):
  """
  y_hat should be a n^L x m matrix
  y should be a n^L x m matrix
  """
  # 1. losses is a n^L x m
  losses = - ( (y * np.log(y_hat)) + (1 - y)*np.log(1 - y_hat) )

  m = y_hat.reshape(-1).shape[0]

  # 2. summing across axis = 1 means we sum across rows,
  #   making this a n^L x 1 matrix
  summed_losses = (1 / m) * np.sum(losses, axis=1)

  # 3. unnecessary, but useful if working with more than one node
  #   in output layer
  return np.sum(summed_losses)

Step 5: Back Propagation

In [134]:
def backprop_layer_3(y_hat, Y, m, A2, W3):
  A3 = y_hat

  # step 1. calculate dC/dZ3 using shorthand we derived earlier
  #     dC/dZ3 = dC/dA3 * dA3/dZ3
  dC_dZ3 = (1/m) * (A3 - Y)
  assert dC_dZ3.shape == (n[3], m)


  # step 2. calculate dC/dW3 = dC/dZ3 * dZ3/dW3
  #   we matrix multiply dC/dZ3 with (dZ3/dW3)^T
  dZ3_dW3 = A2
  assert dZ3_dW3.shape == (n[2], m)

  dC_dW3 = dC_dZ3 @ dZ3_dW3.T
  assert dC_dW3.shape == (n[3], n[2])

  # step 3. calculate dC/db3 = np.sum(dC/dZ3, axis=1, keepdims=True)
  dC_db3 = np.sum(dC_dZ3, axis=1, keepdims=True)
  assert dC_db3.shape == (n[3], 1)

  # step 4. calculate propagator dC/dA2 = dC/dZ3 * dZ3/dA2
  dZ3_dA2 = W3
  dC_dA2 = W3.T @ dC_dZ3
  assert dC_dA2.shape == (n[2], m)

  return dC_dW3, dC_db3, dC_dA2

def backprop_layer_2(propagator_dC_dA2, A1, A2, W2):

  # step 1. calculate dC/dZ2 = dC/dA2 * dA2/dZ2

  # use sigmoid derivation to arrive at this answer:
  #   sigmoid'(z) = sigmoid(z) * (1 - sigmoid(z))
  #     and if a = sigmoid(z), then sigmoid'(z) = a * (1 - a)
  dA2_dZ2 = A2 * (1 - A2)
  dC_dZ2 = propagator_dC_dA2 * dA2_dZ2
  assert dC_dZ2.shape == (n[2], m)


  # step 2. calculate dC/dW2 = dC/dZ2 * dZ2/dW2
  dZ2_dW2 = A1
  assert dZ2_dW2.shape == (n[1], m)

  dC_dW2 = dC_dZ2 @ dZ2_dW2.T
  assert dC_dW2.shape == (n[2], n[1])

  # step 3. calculate dC/db2 = np.sum(dC/dZ2, axis=1, keepdims=True)
  dC_db2 = np.sum(dC_dZ2, axis=1, keepdims=True)
  assert dC_db2.shape == (n[2], 1)

  # step 4. calculate propagator dC/dA1 = dC/dZ2 * dZ2/dA1
  dZ2_dA1 = W2
  dC_dA1 = dZ2_dA1.T @ dC_dZ2
  assert dC_dA1.shape == (n[1], m) # Corrected assertion here

  return dC_dW2, dC_db2, dC_dA1

def backprop_layer_1(propagator_dC_dA1, A1, A0, W1):

  # step 1. calculate dC/dZ1 = dC/dA1 * dA1/dZ1

  # use sigmoid derivation to arrive at this answer:
  #   sigmoid'(z) = sigmoid(z) * (1 - sigmoid(z))
  #     and if a = sigmoid(z), then sigmoid'(z) = a * (1 - a)
  dA1_dZ1 = A1 * (1 - A1)
  dC_dZ1 = propagator_dC_dA1 * dA1_dZ1
  assert dC_dZ1.shape == (n[1], m)


  # step 2. calculate dC/dW1 = dC/dZ1 * dZ1/dW1
  dZ1_dW1 = A0
  assert dZ1_dW1.shape == (n[0], m)

  dC_dW1 = dC_dZ1 @ dZ1_dW1.T
  assert dC_dW1.shape == (n[1], n[0])

  # step 3. calculate dC/db1 = np.sum(dC/dZ1, axis=1, keepdims=True)
  dC_db1 = np.sum(dC_dZ1, axis=1, keepdims=True)
  assert dC_db1.shape == (n[1], 1)

  return dC_dW1, dC_db1

In [135]:
def train():
  # must use global keyword in order to modify global variables
  global W3, W2, W1, b3, b2, b1

  epochs = 10000 # training for 1000 iterations
  alpha = 0.1 # set learning rate to 0.1
  costs = [] # list to store costs

  for e in range(epochs):
    # 1. FEED FORWARD (calculates all A1, A2, A3)
    y_hat, cache = feed_forward(input)

    # 2. COST CALCULATION
    error = cost(y_hat, labels) # Changed cardio to labels
    costs.append(error)

    # 3. BACKPROP CALCULATIONS

    dC_dW3, dC_db3, dC_dA2 = backprop_layer_3(
        y_hat,
        labels, # Changed cardio to labels
        m,
        A2= cache["A2"],
        W3=W3
    )

    dC_dW2, dC_db2, dC_dA1 = backprop_layer_2(
        propagator_dC_dA2=dC_dA2,
        A1=cache["A1"],
        A2=cache["A2"],
        W2=W2
    )

    dC_dW1, dC_db1 = backprop_layer_1(
        propagator_dC_dA1=dC_dA1,
        A1=cache["A1"],
        A0=cache["A0"],
        W1=W1
    )

    # 4. UPDATE WEIGHTS
    W3 = W3 - (alpha * dC_dW3)
    W2 = W2 - (alpha * dC_dW2)
    W1 = W1 - (alpha * dC_dW1)

    b3 = b3 - (alpha * dC_db3)
    b2 = b2 - (alpha * dC_db2)
    b1 = b1 - (alpha * dC_db1)


    if e % 20 == 0:
      print(f"epoch {e}: cost = {error:4f}")

  return costs

In [136]:
costs = train()

epoch 0: cost = 0.959960
epoch 20: cost = 0.697377
epoch 40: cost = 0.691930
epoch 60: cost = 0.691494
epoch 80: cost = 0.691127
epoch 100: cost = 0.690760
epoch 120: cost = 0.690389
epoch 140: cost = 0.690014
epoch 160: cost = 0.689633
epoch 180: cost = 0.689245
epoch 200: cost = 0.688849
epoch 220: cost = 0.688444
epoch 240: cost = 0.688028
epoch 260: cost = 0.687601
epoch 280: cost = 0.687162
epoch 300: cost = 0.686709
epoch 320: cost = 0.686241
epoch 340: cost = 0.685758
epoch 360: cost = 0.685257
epoch 380: cost = 0.684739
epoch 400: cost = 0.684201
epoch 420: cost = 0.683643
epoch 440: cost = 0.683064
epoch 460: cost = 0.682463
epoch 480: cost = 0.681839
epoch 500: cost = 0.681190
epoch 520: cost = 0.680517
epoch 540: cost = 0.679819
epoch 560: cost = 0.679093
epoch 580: cost = 0.678342
epoch 600: cost = 0.677562
epoch 620: cost = 0.676756
epoch 640: cost = 0.675921
epoch 660: cost = 0.675059
epoch 680: cost = 0.674170
epoch 700: cost = 0.673253
epoch 720: cost = 0.672309
epoch 7

In [166]:
import numpy as np

# Creating a new sample input with different values for testing
# Features: age, gender, height, weight, ap_hi, ap_lo, cholesterol, glucose, smokeing, alcohol intake, active
# For example, a younger, healthier individual might have lower risk
# age (in days), gender (1=female, 2=male), height (cm), weight (kg), ap_hi, ap_lo, cholesterol (1=normal, 2=above normal, 3=well above normal), glucose (1=normal, 2=above normal, 3=well above normal), smokeing (0=no, 1=yes), alcohol intake (0=no, 1=yes), active (0=no, 1=yes)

new_sample_input = np.array([18000,	1,	170,	66,	120,	80,	1,	1,	0,	0,	0]).reshape(n[0], 1)

print("New Sample Input Data Shape:", new_sample_input.shape)

# Get prediction using the new sample data
new_prediction, _ = feed_forward(new_sample_input)

print("\nModel Prediction (Probability of Heart Disease) for new sample:")
print(new_prediction)

if new_prediction[0,0] > 0.5:
    print("\nPrediction: High risk of heart disease")
else:
    print("\nPrediction: Low risk of heart disease")

New Sample Input Data Shape: (11, 1)

Model Prediction (Probability of Heart Disease) for new sample:
[[0.62075062]]

Prediction: High risk of heart disease
